In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import json
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://www.soprasteria.no/prosjekter?expertise=all&sector=all")
links

['#readspeakerContent',
 'https://www.soprasteria.no/home',
 '#',
 '#',
 '#',
 '#',
 '#',
 '#',
 '/sok',
 '/',
 'https://www.soprasteria.no/prosjekter/details/digitaliseringsdirektoratet-innovasjon-med-stimulab-tech',
 'https://www.soprasteria.no/prosjekter/details/digitaliseringsdirektoratet-innovasjon-med-stimulab-tech',
 'https://www.soprasteria.no/prosjekter/details/norges-forskningsrad-moderne-datadrevet-svartjeneste-med-servicenow',
 'https://www.soprasteria.no/prosjekter/details/norges-forskningsrad-moderne-datadrevet-svartjeneste-med-servicenow',
 'https://www.soprasteria.no/prosjekter/details/lotteri--og-stiftelsestilsynet-strategisk-plattformvalg-med-salesforce',
 'https://www.soprasteria.no/prosjekter/details/lotteri--og-stiftelsestilsynet-strategisk-plattformvalg-med-salesforce',
 'https://www.soprasteria.no/prosjekter/details/sykehuspartner-samler-kunder-og-leverandrer-i-salesforce',
 'https://www.soprasteria.no/prosjekter/details/sykehuspartner-samler-kunder-og-leverandre

In [11]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure to show the projects the company are working on,
such as links to different projects. The links are dublicated, you only need one
You should respond in JSON as in this example:

{
    "links": [
        {"type": "project", "url": "https://full.url/goes/here/about"},
        {"type": "about", "url": "https://another.full.url/careers"}
    ]
}
"""

In [32]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a company website
 and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
 Respond in markdown without code blocks.
 Include what information they have on their projects and which are using AI.
 
 Also compare it to a candidate profile who has been working in the field of acoustics, and she wants to work with ai. she is caring, driven, ambitious, 
 
 and she wants to thrive at work
 """

In [12]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company projects, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [13]:
print(get_links_user_prompt("https://www.soprasteria.no/prosjekter?expertise=all&sector=all"))


Here is the list of links on the website https://www.soprasteria.no/prosjekter?expertise=all&sector=all -
Please decide which of these are relevant web links for a brochure about the company projects, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#readspeakerContent
https://www.soprasteria.no/home
#
#
#
#
#
#
/sok
/
https://www.soprasteria.no/prosjekter/details/digitaliseringsdirektoratet-innovasjon-med-stimulab-tech
https://www.soprasteria.no/prosjekter/details/digitaliseringsdirektoratet-innovasjon-med-stimulab-tech
https://www.soprasteria.no/prosjekter/details/norges-forskningsrad-moderne-datadrevet-svartjeneste-med-servicenow
https://www.soprasteria.no/prosjekter/details/norges-forskningsrad-moderne-datadrevet-svartjeneste-med-servicenow
https://www.soprasteria.no/prosjekter/details/lotteri--og-stiftelsestilsynet-strategisk-plattformvalg-med-salesforce
https://www.soprasteria.no/prosj

In [14]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [15]:
select_relevant_links("https://www.soprasteria.no/prosjekter?expertise=all&sector=all")


KeyboardInterrupt: 

In [19]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its page that overview projects and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks to present their projects.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    
    #user_prompt += fetch_website_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("Sopra Steria", "https://www.soprasteria.no/prosjekter?expertise=all&sector=all")

TypeError: can only concatenate str (not "list") to str

In [27]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [34]:
create_brochure("Sopra Steria", "https://www.soprasteria.no/prosjekter?expertise=kunstig-intelligens&sector=all")

# Sopra Steria: Navigating the Future with Innovation, AI & a Dash of Magic

## Who Are We?  
Sopra Steria is that clever brainiac friend everyone wishes they had — a global powerhouse born from the merger of cutting-edge technology and big-hearted ambition. From Norway to Singapore, from finance to health care, we help organizations *not* just keep up with digital transformation but lead the pack — all while having a bit of fun. 

Want to be part of something that sparks innovation, drives public good, and isn’t afraid to dance with AI? Let’s talk.

---

## Our Projects: A Symphony of Smart Solutions and AI-Powered Brilliance

### 1. **Digitaliseringsdirektoratet: Innovasjon med StimuLab Tech**  
**Sector:** Public | **Focus:** Innovation & Artificial Intelligence  
Public meets startup magic in a project that throws out the old rulebook on how government services innovate. Despite the hefty bureaucracy and risk-aversion in the public sector, Sopra Steria shakes things up by linking government institutions with scrappy startups. The result? Responsible, groundbreaking AI-powered solutions co-created to solve real societal challenges. Think of it as matchmaking for tech progress, with Sopra as the cupid.

### 2. **Norges forskningsråd: Moderne, datadrevet svartjeneste med ServiceNow**  
**Sector:** Public | **Focus:** AI, IT Systems Development & Security  
A smart, data-driven support service wrapped in the sleek ServiceNow platform, designed to offer a one-stop, professional user experience. Behind the scenes lurks AI to streamline, analyze, and elevate user support across Norway’s vibrant research landscape—making it easier for users to get help and keep moving forward.

### 3. **Gassco: Holder Europa varm**  
**Sector:** Energy | **Focus:** Security & IT Systems  
Powering warmth for millions, Gassco relies on Sopra Steria to keep their IT secure and operational 24/7 — because cold is the enemy, and uptime is king.

### 4. **Archer: IT That Works Globally 24/7**  
**Sector:** Energy | **Focus:** Cloud & IT Systems  
Global energy demands don't sleep, and neither does Archer’s IT. Sopra Steria delivers cloud-based, always-on systems ensuring Archer’s worldwide operations flow smoothly.

### 5. **Statnett: Empowering the Energy Grid with Smart Data Analysis**  
**Sector:** Energy | **Focus:** Innovation  
With electrification on the rise, Statnett leverages Sopra Steria's innovative data analytics to make sense of the sprawling power grid data — boosting efficiency and powering Norway’s green transition.

### 6. **Kongsberg Digital: Cloud-Based Navigation for Maritime Simulator Training**  
**Sector:** Defense & National Security | **Focus:** Innovation & Cloud  
Sailing or simulating? Sopra Steria crafts cloud solutions for navigation training simulators, prepping maritime professionals to conquer the seas with tech on their side.

### 7. **Ruter: Inclusive Transport Experience**  
**Sector:** Transport | **Focus:** User Experience & Innovation  
Striving for accessible transport? Sopra Steria designs inclusive, user-friendly transport services to ensure everyone gets a smooth ride—no exceptions.

---

## Why Sopra Steria?  

- **We don’t just do AI; we infuse it into real-world challenges.** Artificial intelligence isn’t a buzzword here; it’s the heartbeat of many projects, from public sector transformations to energy innovations.  
- **Deep sector knowledge mixed with innovation mojo.** Whether it's healthcare, energy, finance, or government, Sopra Steria knows the landscape and how to innovate inside it.  
- **Global footprint with a local touch.** Your brilliant ideas get the global scale + the local sensitivity needed to succeed.  
- **A culture of collaboration.** We bring startups and public bodies to dance the innovation tango—and yes, we lead.  

---

## For the Ambitious Acoustics Expert Eyeing AI:  

Imagine you — driven, caring, ambitious, ready to thrive — joining forces with Sopra Steria. Your acoustics know-how is a powerhouse waiting to supercharge artificial intelligence projects. Here, AI isn’t just codes and algorithms, it’s about building **better, smarter, more human** solutions. Whether integrating AI into user experiences, energy data analytics, or health sector service innovation, your passion finds room to roar.

- You'll love the **impact-driven work**, mixing tech innovation and societal good.  
- The **collaborative environment** fits your caring, team-first mindset.  
- Your **ambition meets opportunity** with projects that reward curiosity and courage.  
- Here, thriving is a norm, not a buzzword—growth, challenge, and fun are all stocked on the menu.  

Ready to switch your frequency and ride the AI wave? Sopra Steria’s got the amps—just bring your spark.

---

## Snooping to join?  
Visit [Sopra Steria Careers](https://www.soprasteria.no/bli-en-av-oss) and jump into a role where innovation, AI, and your own ambition harmonize into a career hit parade.

---

**Sopra Steria**  
*Where technology meets heart. Because we believe the future deserves both.*

In [33]:
create_brochure("HuggingFace", "https://huggingface.co")

# Welcome to Hugging Face: Where AI Meets Community and Innovation!

---

## Who Are We?

Hugging Face is not just a tech company; it’s a vibrant, buzzing hive of **2 million+ AI models, datasets, and apps** where the global machine learning community **creates, shares, and collaborates** to build the AI of tomorrow. Whether you speak text, image, audio, or even 3D, Hugging Face has you covered.

If Open Source AI were a party, we’re the lively dance floor—and everyone’s invited!

---

## Our AI Playground: Projects & Innovations

### Models Galore  
Browse an astonishing **2 million+ models** crafted for tasks like:
- Text Generation (think GPT but with flair)  
- Image-to-Text and Text-to-Image (your visual creativity unleashed)  
- Text-to-Speech and Audio-to-Audio (even your ears have AI pals)  
- Any-to-Any - the Swiss Army knife of AI transformations  
- And many more with parameter sizes from tiny sprites (under 1 billion) to mammoth giants (over 500 billion)!

**Trending models include:**

- **zai-org/GLM-5** (a 754B-parameter text generator — fast, fierce, fabulous)  
- **MiniMaxAI/MiniMax-M2.5** (229B text generation magicien)  
- **Qwen3.5-397B-A17B** (image-text-to-text; your AI storyteller)  
- **Nanbeige4.1-3B** (small but mighty text generator)  
- **moonshotai/Kimi-K2.5** (image-text-to-text with a whopping 171B parameters)  

The whole spectrum of AI is here—from playful mini models to industrial-strength engines ready for enterprise.

---

### Datasets: The Fuel for AI Magic

A staggering **over 850,000+ datasets** support learning and experimentation, making sure your AI is well-fed. Among the hottest:

- **openbmb/UltraData-Math:** 181 million rows of math problems and concepts  
- **ma-xu/fine-t2i:** Nearly a million samples to fine-tune text-to-image dreams  
- **OpenMed/Medical-Reasoning-SFT-Mega:** Specialized medical reasoning datasets  

**Formats?** JSON, CSV, parquet, imagefolder—you name it. Modalities? Text, audio, video, geospatial, 3D—you get the picture.

---

### Spaces: The AI Fun Zone for Apps & Demos

Spaces host over **700+ live AI apps and demos** you can try right now, for free. Some fan favorites include:

- **Wan2.2 Animate:** Generate video animations from images and text prompts  
- **ACE-Step v1.5:** A music generation foundation model to make your ears dance  
- **Reachy Phone Home:** A quirky phone-focus companion for the Reachy robot

You build, we host—show the world your AI brilliance!

---

### Enterprise & Compute: For Teams That Mean Business

From startups to large enterprises, Hugging Face offers **paid compute, dedicated support, and enterprise-grade security** to rocket your AI projects safely and swiftly into production.

Starts at $2 (less than your daily coffee)—because great ML should be accessible to all.

---

## Hugging Face vs. Your Candidate Profile: The Perfect Match

Meet our ideal recruit: 

- Experienced in **acoustics** and eager to delve into AI—check!  
- Caring and driven? Hugging Face thrives on collaboration and community, where empathy and teamwork matter.  
- Ambitious with a hunger to thrive at work? The fast-paced development of new models and datasets means there’s always a new learning curve, a fresh challenge, and a chance to lead cutting-edge projects.  
- Interested in audio? Hugging Face’s **audio-to-audio** and **text-to-speech models** are just waiting for someone who understands sound at a deep level to push them even further.

If she loves accelerating AI *and* wants a place to grow with open arms, Hugging Face is where she can **harmonize her acoustics skills with state-of-the-art AI innovation**.

---

## Why Join or Invest?

- Access to a thriving **community of top AI developers worldwide**  
- Tools and infrastructure that scale from **hobbyist playground to enterprise powerhouse**  
- Open source driven—but with serious enterprise chops  
- A platform that respects and fosters diversity, creativity, and rapid innovation  

---

## So, Are You Ready to Join the Future of AI?

Jump in at **huggingface.co** and become part of a community where your **ambition meets collaboration**, your **caring spirit fuels innovation**, and your **AI aspirations can roar to life**.

---

### Hugging Face

*Building the Future, One Model at a Time—with a Smile! 🤗*